# Configurations

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
import sklearn.decomposition
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer
import seaborn as sns; sns.set()
import math
from typing import List
from types import SimpleNamespace
from sklearn.model_selection import train_test_split

# Temporary
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import copy
import os
os.chdir("..")

# Data Preprocessing

In [12]:
# The column names in the NSL KDD dataset
col_names = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login",
    "count","srv_count","serror_rate","srv_serror_rate","rerror_rate",
    "srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
    "dst_host_count","dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label",               # <-- attack label
    "difficulty"           # <-- the missing column!
]

train = pd.read_csv("data/KDDTrain+.txt", names=col_names)
test  = pd.read_csv("data/KDDTest+.txt",  names=col_names)

# Selected features
features = [
    # categorical
    "protocol_type", "service", "flag",

    # numeric
    "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "dst_host_srv_count", "dst_host_serror_rate",
    "logged_in", "root_shell", "su_attempted"
]

categorical = ["protocol_type", "service", "flag"]

numeric = [
    "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "dst_host_srv_count", "dst_host_serror_rate",
    "logged_in", "root_shell", "su_attempted"
]

preprocess = ColumnTransformer([ ("cat", OneHotEncoder(handle_unknown="ignore"), categorical), ("num", "passthrough", numeric) ])

# Training and testing set
X_train = train[features]
X_test  = test[features]

# Ensure string values are numerical
X_train = preprocess.fit_transform(X_train).toarray()
X_test  = preprocess.transform(X_test).toarray()

# Label normal traffic as 0 and attack traffic 1
# y_train = (train["label"] != "normal").astype(int).values
# y_test  = (test["label"] != "normal").astype(int).values

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Fit on ALL labels so no unseen labels appear later
all_labels = pd.concat([train["label"], test["label"]], axis=0)
le.fit(all_labels)

# Now safely transform
y_train_full = le.transform(train["label"])
y_test       = le.transform(test["label"])

num_classes = len(le.classes_)
print("Number of classes:", num_classes)


# Shape of data
num_features = X_train.shape[1] # 94 features

# X_train is already preprocessed earlier
N = len(X_train)
val_size = int(0.2 * N)

# Apply the same split to encoded labels
y_val   = y_train_full[:val_size]
y_train = y_train_full[val_size:]

X_val   = X_train[:val_size]
X_train = X_train[val_size:]

Number of classes: 40


In [13]:
print(X_test.shape)

(22544, 94)


In [16]:
# stratification
X_full = np.concatenate([X_train, X_val, X_test], axis=0)
y_full = np.concatenate([y_train, y_val, y_test], axis=0)


X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=0.2,
    stratify=y_full,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

# Neural Networks

In [4]:
class Layer:
    def __init__(self, input_size: int, output_size: int):
        # Weight matrix uses He initialization
        self.weight = np.random.randn(output_size, input_size) * np.sqrt(2 / input_size)
        
        # Bias vector initialized to zero
        self.bias = np.zeros((output_size, 1))

In [5]:
def generate_single_layer(input_size: int, output_size: int):
  """
  Generate a single layer with .weight and .bias attributes
  """

  return Layer(input_size, output_size)

In [6]:
def generate_layers(layers_list: List, L=3):
  """
  Generate layers with biases and weights as per a given list
  """
  layers = [None]  # Index 0 unused
  for i in range(len(layers_list) - 1):
        input_size, output_size  = layers_list[i], layers_list[i + 1]
        layers.append(generate_single_layer(input_size, output_size))
  return layers

In [7]:
def relu(z):
    """ReLU activation."""
    return np.maximum(0, z)

def relu_derivative(z):
    """Derivative of ReLU."""
    return (z > 0).astype(float)


def softmax(z):
    """Softmax for column-vector logits."""
    z_shift = z - np.max(z)          # numerical stability
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z)

def cross_entropy_loss(pred, target):
    """Cross‑entropy loss for one sample."""
    return -np.sum(target * np.log(pred + 1e-12))


def grad_output_layer(y_true, y_pred):
    """
    Gradient at output layer for softmax + cross‑entropy.
    The derivative simplifies beautifully to:
        g = y_pred - y_true
    No need for softmax derivative.
    """
    return y_pred - y_true


def preactivation(W, b, a_prev):
    """Compute z = W a_prev + b."""
    return W @ a_prev + b


def activation(z):
    """Hidden layer activation: ReLU."""
    return relu(z)


def grad_hidden_layer(g_next, W_next, z):
    """
    Gradient at hidden layer:
        g_l = (W_{l+1}^T g_{l+1}) ⊙ relu'(z_l)
    """
    return (W_next.T @ g_next) * relu_derivative(z)


def grad_weight(g, a_prev):
    """Gradient wrt weights."""
    return g @ a_prev.T


def grad_bias(g):
    """Gradient wrt bias."""
    return g

In [20]:
def train_neural_network(layers, layers_list, L, lr, X, Y, weight_decay = 1e-4):
    for x, y in zip(X, Y):

        # Handle data
        x = x.reshape(-1, 1)

        # Convert scalar label → one‑hot column vector
        y_vec = np.zeros((layers[-1].weight.shape[0], 1))
        y_vec[y] = 1.0

        # Initialize for forward pass
        activations = [x]
        zs = [None]

        # Forward pass in hidden layers uses ReLu
        for layer in layers[1:-1]:
            z = preactivation(layer.weight, layer.bias, activations[-1])
            a = activation(z)                  
            zs.append(z)
            activations.append(a)

        # Forward pass in final layer uses softmax
        last = layers[-1]
        z = preactivation(last.weight, last.bias, activations[-1])
        a = softmax(z)
        zs.append(z)
        activations.append(a)

        # Initialize for backpropogation
        g = [None]
        nabla_W = [None]
        nabla_b = [None]

        g.extend([np.zeros((m, 1)) for m in layers_list])
        nabla_W.extend([np.zeros_like(layer.weight) for layer in layers[1:]])
        nabla_b.extend([np.zeros_like(layer.bias) for layer in layers[1:]])

        # Backpropogation
        for l in range(L - 1, 0, -1):

            # Last layer gradient - softmax + cross‑entropy
            if l == L - 1:
                g[l] = grad_output_layer(y_vec, activations[l])
            
            # Hidden layers
            else:
                g[l] = grad_hidden_layer(g[l + 1], layers[l + 1].weight, zs[l])

            nabla_W[l] = grad_weight(g[l], activations[l - 1])
            nabla_b[l] = grad_bias(g[l])

        # Update
        for l in range(L - 1, 0, -1):
            layers[l].weight -= lr * (nabla_W[l]+ weight_decay*layers[l].weight)
            layers[l].bias   -= lr * nabla_b[l]

In [9]:
def forward(layers, x):
    """
    Forward propagation, calculating activation of the final layer
    """
    a = x
    for layer in layers[1:]:
        a = softmax(layer.bias + layer.weight @ a)
    return a

def prediction(layers, X, Y):
    correct = 0
    total = len(X)

    for (x, y) in zip(X, Y):
        x = x.reshape(-1, 1)
        probs = forward(layers, x)

        pred =  np.argmax(probs)
        if pred == y:
            correct += 1

    accuracy = correct / total * 100
    return accuracy

# Generalization

## Grid Search

In [21]:
# Layers
layers_list = [num_features, 67, 40]
lrs = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
weight_decays = [0, 1e-5, 3e-5, 1e-4, 3e-4]

best_acc = -1
best_lr = None
best_layers = None

# Grid search over learning rate
for lr in lrs:
    for wd in weight_decays:
        # Recreate layers
        layers = generate_layers(layers_list)

        # Train model with learning rate lr
        train_neural_network(layers, layers_list, len(layers_list), lr, X_train, y_train, weight_decay=wd)

        # Compute validation accuracy
        val_acc = prediction(layers, X_val, y_val)

        # Keep best hyperparameters
        if val_acc > best_acc:
            best_acc, best_lr,best_layers, best_wd = val_acc, lr, copy.deepcopy(layers), wd

print("Best Learning Rate:", best_lr)
print("Best weight decay:", best_wd)
print("Validation Accuracy:", best_acc)

Best Learning Rate: 0.0001
Validation Accuracy: 82.17396793334176


## Evaluation

In [22]:
train_acc = prediction(best_layers, X_train, y_train)
test_acc = prediction(best_layers, X_test, y_test)
print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 82.13361388742767
Test Accuracy: 82.10342041475896
